In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from trace_comparison.training.classifier import load_classifier
from utils.notebook_utils import reset_cwd_and_env

reset_cwd_and_env()

BASEPATH = Path.cwd()

# ---- 1. Load the checkpoint -------------------------------------------------
CKPT = BASEPATH / "checkpoints" / "classifier" / "02.pt"   # <- point at your run
device = "cuda" if torch.cuda.is_available() else "cpu"

model, classes = load_classifier(CKPT, device=device)
classes = list(classes)
print(f"loaded {CKPT.name} with {len(classes)} classes on {device}: {classes}")

payload = torch.load(CKPT, map_location="cpu", weights_only=True)
print("train metrics:", payload.get("metrics"))
print("val confusion shape:", tuple(payload["val_confusion"].shape))

# ---- 2. Class-weight cosine similarity (final Linear layer) -----------------
# The last layer is Linear(hidden -> n_classes); each row is a class prototype.
W = model.net[-1].weight.detach().cpu()              # (C, H)
Wn = W / W.norm(dim=1, keepdim=True).clamp_min(1e-12)
cos = (Wn @ Wn.T).numpy()                            # (C, C) in [-1, 1]

# Pretty print
header = " " * 14 + "  ".join(f"{c[:8]:>8}" for c in classes)
print(header)
for i, ci in enumerate(classes):
    print(f"{ci:<14}" + "  ".join(f"{cos[i, j]:+8.3f}" for j in range(len(classes))))

# ---- 3. Heatmap -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(1.0 * len(classes) + 2, 1.0 * len(classes) + 1.2))
im = ax.imshow(cos, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right")
ax.set_yticks(range(len(classes)), classes)
ax.set_title("Cosine similarity of class weight vectors")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

# ---- 4. Most / least similar class pairs (off-diagonal) ---------------------
C = len(classes)
iu = np.triu_indices(C, k=1)
pairs = sorted(
    ((cos[i, j], classes[i], classes[j]) for i, j in zip(*iu)),
    reverse=True,
)
print("\nMost similar class pairs:")
for s, a, b in pairs[:5]:
    print(f"  {a:<10} vs {b:<10}  {s:+.3f}")
print("Least similar class pairs:")
for s, a, b in pairs[-5:]:
    print(f"  {a:<10} vs {b:<10}  {s:+.3f}")